# MLflow 실험 관리

MLflow는 ML 실험을 추적하고 관리하는 도구입니다.

## 학습 목표
- 실험 추적 (Tracking)
- 모델 저장/로딩
- 모델 레지스트리
- 배포

In [ ]:
import mlflow
import mlflow.sklearn
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import numpy as np

print(f"MLflow 버전: {mlflow.__version__}")

## 1. 기본 실험 추적

In [ ]:
# 데이터 로딩
iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.2, random_state=42
)

# MLflow 실험 설정
mlflow.set_experiment("Iris Classification")

In [ ]:
# 실험 실행
with mlflow.start_run(run_name="random_forest_v1"):
    # 하이퍼파라미터
    n_estimators = 100
    max_depth = 10
    
    # 모델 학습
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )
    model.fit(X_train, y_train)
    
    # 예측 및 평가
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    
    # 파라미터 기록
    mlflow.log_param("n_estimators", n_estimators)
    mlflow.log_param("max_depth", max_depth)
    
    # 메트릭 기록
    mlflow.log_metric("accuracy", accuracy)
    
    # 모델 저장
    mlflow.sklearn.log_model(model, "model")
    
    print(f"정확도: {accuracy:.4f}")
    print("MLflow에 실험 기록 완료!")

## 2. 여러 실험 비교

In [ ]:
# 여러 하이퍼파라미터로 실험
param_sets = [
    {"n_estimators": 50, "max_depth": 5},
    {"n_estimators": 100, "max_depth": 10},
    {"n_estimators": 200, "max_depth": 20},
    {"n_estimators": 100, "max_depth": None}
]

for i, params in enumerate(param_sets):
    with mlflow.start_run(run_name=f"experiment_{i+1}"):
        model = RandomForestClassifier(**params, random_state=42)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        accuracy = accuracy_score(y_test, y_pred)
        
        mlflow.log_params(params)
        mlflow.log_metric("accuracy", accuracy)
        mlflow.sklearn.log_model(model, "model")
        
        print(f"실험 {i+1}: {params} -> 정확도: {accuracy:.4f}")

## 3. 실험 결과 조회

In [ ]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

# 실험 목록 조회
experiments = client.list_experiments()
print("실험 목록:")
for exp in experiments:
    print(f"  - {exp.name} (ID: {exp.experiment_id})")

In [ ]:
# 가장 좋은 실행 찾기
experiment = client.get_experiment_by_name("Iris Classification")
runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.accuracy DESC"],
    max_results=1
)

best_run = runs[0]
print(f"최고 정확도: {best_run.data.metrics['accuracy']:.4f}")
print(f"파라미터: {best_run.data.params}")

## 4. 모델 레지스트리

In [ ]:
# 모델을 레지스트리에 등록
model_uri = f"runs:/{best_run.info.run_id}/model"
model_name = "iris-classifier"

# 모델 등록
mlflow.register_model(model_uri, model_name)
print(f"모델 '{model_name}' 등록 완료!")

In [ ]:
# 레지스트리에서 모델 로딩
loaded_model = mlflow.pyfunc.load_model(f"models:/{model_name}/Staging")

# 예측
sample = X_test[:1]
prediction = loaded_model.predict(sample)
print(f"예측 결과: {prediction}")

## 5. MLflow UI 실행

터미널에서 다음 명령어로 MLflow UI를 실행할 수 있습니다:

```bash
mlflow ui
```

기본적으로 `http://localhost:5000`에서 실행됩니다.

## MLflow 주요 기능 요약

| 기능 | 설명 |
|------|------|
| Tracking | 실험 파라미터, 메트릭, 모델 기록 |
| Projects | 재현 가능한실험 환경 |
| Models | 다양한 프레임워크 모델 저장/로딩 |
| Model Registry | 모델 버전 관리, 승인 프로세스 |
| UI | 웹 기반 실험 대시보드 |